In [2]:
import cv2
import pandas as pd
import numpy as np
import os

# --- CONFIGURATION ---
PARQUET_FILE = "lip_products_final_colors_full.parquet"
OUTPUT_FILE = "lip_products_labeled.parquet"
IMAGE_DIR = "images"

def create_display_image(img_path, candidates, current_choice_idx=None):
    """Crée une image composite : Photo à gauche, Candidats à droite."""
    # 1. Charger et redimensionner l'image source
    img = cv2.imread(img_path)
    if img is None: return None
    
    # On resize pour que ça tienne bien à l'écran (Hauteur 400px)
    h, w = img.shape[:2]
    target_h = 400
    ratio = target_h / h
    img = cv2.resize(img, (int(w * ratio), target_h))
    
    # 2. Créer la bande des candidats
    swatch_w = 150
    candidates_img = np.zeros((target_h, swatch_w, 3), dtype=np.uint8)
    
    # On divise la hauteur par le nombre de candidats (3 normalement)
    step_h = target_h // len(candidates)
    
    for i, color in enumerate(candidates):
        # Couleur (OpenCV est en BGR, le fichier est stocké en RGB -> conversion)
        bgr_color = [int(color[2]), int(color[1]), int(color[0])]
        
        # Dessiner le rectangle
        y_start = i * step_h
        y_end = (i + 1) * step_h
        cv2.rectangle(candidates_img, (0, y_start), (swatch_w, y_end), bgr_color, -1)
        
        # Texte (Choix 1, 2, 3)
        cv2.putText(candidates_img, f"Touche {i+1}", (10, y_start + 40), 
                    cv2.FONT_HERSHEY_SIMPLEX, 0.8, (255, 255, 255), 2)
        cv2.putText(candidates_img, f"Touche {i+1}", (12, y_start + 42), 
                    cv2.FONT_HERSHEY_SIMPLEX, 0.8, (0, 0, 0), 2)

    # 3. Coller les deux
    composite = np.hstack((img, candidates_img))
    return composite

def run_labeler():
    # Chargement des données
    if os.path.exists(OUTPUT_FILE):
        print(f"🔄 Reprise du fichier existant : {OUTPUT_FILE}")
        df = pd.read_parquet(OUTPUT_FILE)
    else:
        print(f"🆕 Nouveau fichier de travail basé sur : {PARQUET_FILE}")
        df = pd.read_parquet(PARQUET_FILE)
        # On crée la colonne de validation si elle n'existe pas
        if 'manual_label' not in df.columns:
            df['manual_label'] = None # Stockera la couleur finale [R, G, B]
        if 'label_status' not in df.columns:
            df['label_status'] = 'todo' # 'done', 'ignore'

    # Filtrer ce qui reste à faire
    # On ne traite que ce qui n'est pas encore 'done' ou 'ignore'
    todo_indices = df[df['label_status'] == 'todo'].index
    
    print(f"🎯 Il reste {len(todo_indices)} images à valider.")
    print("-------------------------------------------------")
    print("COMMANDES CLAVIER :")
    print(" [1], [2], [3] : Choisir la couleur correspondante")
    print(" [X]           : Jeter (Image impossible/pas de bonne couleur)")
    print(" [S]           : Passer (Skip) pour y revenir plus tard")
    print(" [ESC] ou [Q]  : Sauvegarder et Quitter")
    print("-------------------------------------------------")

    count = 0
    save_frequency = 10 # Sauvegarde auto toutes les 10 images

    for idx in todo_indices:
        row = df.loc[idx]
        img_path = os.path.join(IMAGE_DIR, str(row['image_filename']))
        centers = row['kmeans_centers']
        
        if centers is None or len(centers) == 0:
            df.at[idx, 'label_status'] = 'ignore'
            continue

        # Affichage
        display_img = create_display_image(img_path, centers)
        if display_img is None:
            df.at[idx, 'label_status'] = 'ignore'
            continue
            
        cv2.imshow("Labeler (Q to Quit)", display_img)
        
        valid_choice = False
        while not valid_choice:
            key = cv2.waitKey(0)
            
            # --- LOGIQUE DE CHOIX ---
            
            # Touche '1' (Code ASCII 49) -> Premier candidat
            if key == ord('1') and len(centers) >= 1:
                df.at[idx, 'manual_label'] = centers[0]
                df.at[idx, 'label_status'] = 'done'
                valid_choice = True
                print(f"✅ Image {idx}: Choix 1")

            # Touche '2' (Code ASCII 50) -> Deuxième candidat
            elif key == ord('2') and len(centers) >= 2:
                df.at[idx, 'manual_label'] = centers[1]
                df.at[idx, 'label_status'] = 'done'
                valid_choice = True
                print(f"✅ Image {idx}: Choix 2")

            # Touche '3' (Code ASCII 51) -> Troisième candidat
            elif key == ord('3') and len(centers) >= 3:
                df.at[idx, 'manual_label'] = centers[2]
                df.at[idx, 'label_status'] = 'done'
                valid_choice = True
                print(f"✅ Image {idx}: Choix 3")
            
            # Touche 'x' -> Rejeter
            elif key == ord('x') or key == ord('X'):
                df.at[idx, 'label_status'] = 'ignore'
                valid_choice = True
                print(f"🗑️ Image {idx}: Rejetée")

            # Touche 's' -> Skip
            elif key == ord('s') or key == ord('S'):
                valid_choice = True # On sort de la boucle while, mais on laisse le status 'todo'
                print(f"⏭️ Image {idx}: Passée")

            # Quitter
            elif key == 27 or key == ord('q'): # ESC or q
                print("💾 Sauvegarde et fermeture...")
                df.to_parquet(OUTPUT_FILE)
                cv2.destroyAllWindows()
                return

        count += 1
        # Sauvegarde automatique
        if count % save_frequency == 0:
            df.to_parquet(OUTPUT_FILE)
            print("--- (Sauvegarde Auto) ---")

    # Sauvegarde finale si boucle terminée
    df.to_parquet(OUTPUT_FILE)
    cv2.destroyAllWindows()
    print("🎉 Session terminée !")

# Lancer l'outil
if __name__ == "__main__":
    run_labeler()

🔄 Reprise du fichier existant : lip_products_labeled.parquet
🎯 Il reste 14630 images à valider.
-------------------------------------------------
COMMANDES CLAVIER :
 [1], [2], [3] : Choisir la couleur correspondante
 [X]           : Jeter (Image impossible/pas de bonne couleur)
 [S]           : Passer (Skip) pour y revenir plus tard
 [ESC] ou [Q]  : Sauvegarder et Quitter
-------------------------------------------------
✅ Image 33527: Choix 1
✅ Image 33528: Choix 2
✅ Image 33529: Choix 3
✅ Image 33530: Choix 1
✅ Image 33531: Choix 3
💾 Sauvegarde et fermeture...


In [3]:
df_Lip_test_labeled = pd.read_parquet("lip_products_labeled.parquet")
df_Lip_test_labeled.info()
df_Lip_test_labeled[['rgb_extracted', 'kmeans_centers', 'manual_label', 'label_status']]

<class 'pandas.core.frame.DataFrame'>
Index: 14634 entries, 33523 to 48156
Data columns (total 16 columns):
 #   Column                 Non-Null Count  Dtype 
---  ------                 --------------  ----- 
 0   product_id             14634 non-null  object
 1   cluster_id             14634 non-null  Int64 
 2   country_name           14634 non-null  object
 3   title                  14634 non-null  object
 4   description            11983 non-null  object
 5   category_level_1_name  14634 non-null  object
 6   category_level_2_name  14634 non-null  object
 7   category_level_3_name  14634 non-null  object
 8   brand_name             14634 non-null  object
 9   url                    14634 non-null  object
 10  shade_name             14393 non-null  object
 11  image_filename         14634 non-null  object
 12  rgb_extracted          13855 non-null  object
 13  kmeans_centers         13855 non-null  object
 14  manual_label           9 non-null      object
 15  label_status        

,rgb_extracted,kmeans_centers,manual_label,label_status
33523,"[118, 45, 81]","[[153, 84, 117], [118, 45, 81], [204, 161, 171]]","[118, 45, 81]",done
33524,"[214, 128, 135]","[[214, 128, 135], [227, 153, 159], [237, 190, ...","[227, 153, 159]",done
33525,"[231, 65, 108]","[[231, 65, 108], [238, 175, 188], [217, 118, 1...","[231, 65, 108]",done
33526,"[156, 88, 92]","[[215, 125, 135], [219, 168, 172], [156, 88, 92]]","[215, 125, 135]",done
33527,"[131, 32, 45]","[[131, 32, 45], [187, 150, 153], [160, 89, 93]]","[131, 32, 45]",done
...,...,...,...,...
48152,None,None,None,todo
48153,None,None,None,todo
48154,None,None,None,todo
48155,"[211, 23, 105]","[[211, 23, 105], [216, 151, 173], [210, 64, 118]]",None,todo
